# Adversarial Examples & Robustness

## 1. Introduction

Neural networks can be surprisingly fragile. A tiny, imperceptible change to an input image can completely fool a model that was performing near-perfectly. These carefully crafted perturbations are called **adversarial examples**.

### What we'll learn:
- How to generate adversarial examples using **FGSM** and **PGD** attacks
- Why neural networks are vulnerable to these attacks
- How to defend against them through **adversarial training**
- The theory of **certified defenses** that provide provable robustness
- Why this matters critically for real-world deployment

### Why this matters:
Adversarial examples aren't just an academic curiosity. They represent a fundamental security vulnerability in deployed ML systems:
- **Autonomous vehicles** could misclassify stop signs with carefully placed stickers
- **Face recognition systems** could be fooled by special glasses or makeup
- **Malware detectors** could be evaded with tiny file modifications
- **Medical diagnosis** systems could make dangerous errors

Understanding adversarial robustness is essential for deploying safe, reliable ML systems.

## 2. Setup

First, let's import our dependencies and configure the environment.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from aiml_notebooks import (
    get_device,
    set_seed,
    count_parameters,
    MNIST_MEAN,
    MNIST_STD,
)

### Set random seed for reproducibility

This ensures our experiments produce consistent results.

In [ ]:
set_seed(42)

### Configure device

We'll use GPU acceleration if available for faster training and attack generation.

In [ ]:
device = get_device()

## 3. Dataset & Model Preparation

We'll use MNIST for this demonstration because it's fast to train and makes visualizations easy to interpret. The same techniques work for any image dataset (CIFAR-10, ImageNet, etc.).

### Load MNIST dataset

We'll normalize the images to have zero mean and unit variance, which improves training stability.

In [ ]:
# Transforms: convert to tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

# Load train and test datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

### Define a simple CNN model

We'll build a standard convolutional neural network with:
- Two convolutional layers with ReLU activations and max pooling
- Two fully connected layers
- Dropout for regularization

This architecture achieves ~99% accuracy on MNIST.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 14x14 -> 14x14
        self.pool = nn.MaxPool2d(2, 2)  # Halves spatial dimensions
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Conv block 1: Conv -> ReLU -> Pool
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
        
        # Conv block 2: Conv -> ReLU -> Pool
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7
        
        # Flatten
        x = x.view(-1, 64 * 7 * 7)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

model = SimpleCNN().to(device)
print(f"Model parameters: {count_parameters(model):,}")

### Train the model

We'll train for just 3 epochs to get reasonable accuracy (~98-99%). This is enough to demonstrate adversarial vulnerabilities.

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, device):
    """Evaluate model accuracy."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return 100. * correct / total

### Run the training loop

We'll use Adam optimizer and cross-entropy loss.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    test_acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {train_loss:.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

print(f"\nFinal test accuracy: {test_acc:.2f}%")

## 4. Understanding Adversarial Examples

Before generating attacks, let's understand the intuition behind adversarial examples.

### The key insight:
Neural networks make predictions by computing weighted sums of inputs. Even though individual pixel changes are imperceptible to humans, if we change **many pixels slightly** in a **coordinated direction**, the cumulative effect on the model's internal computations can be huge.

Think of it like this:
- A neural network is a complex function: `f(x) = output`
- The **gradient** `∇f(x)` tells us how to change `x` to increase/decrease the output
- We can use gradients to find the **most efficient way** to fool the model

This is fundamentally different from random noise, which affects all directions equally and is easily filtered out by the network's nonlinearities.

### Visualize model predictions on clean examples

Let's first see how the model performs on normal images.

In [ ]:
def denormalize(tensor):
    """Denormalize MNIST images for visualization."""
    return tensor * MNIST_STD + MNIST_MEAN

def show_images(images, labels, predictions, title):
    """Display a grid of images with labels and predictions."""
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    for idx, ax in enumerate(axes.flat):
        if idx < len(images):
            img = denormalize(images[idx].cpu()).squeeze()
            ax.imshow(img, cmap='gray')
            color = 'green' if labels[idx] == predictions[idx] else 'red'
            ax.set_title(f"True: {labels[idx]}, Pred: {predictions[idx]}", color=color)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Get a batch of test images
test_images, test_labels = next(iter(test_loader))
test_images, test_labels = test_images.to(device), test_labels.to(device)

# Get model predictions
model.eval()
with torch.no_grad():
    outputs = model(test_images)
    _, predictions = outputs.max(1)

# Show first 10 examples
show_images(test_images[:10], test_labels[:10].cpu().numpy(), 
            predictions[:10].cpu().numpy(), "Clean Examples (Original Images)")

## 5. FGSM: Fast Gradient Sign Method

**FGSM** (Fast Gradient Sign Method) is the simplest and fastest adversarial attack. It was introduced by Goodfellow et al. in 2015.

### The algorithm:
Given an image `x` with true label `y`:

1. Compute the loss: `L(θ, x, y)` where `θ` are model parameters
2. Compute gradient with respect to the **input**: `∇ₓ L(θ, x, y)`
3. Create adversarial example: `x_adv = x + ε · sign(∇ₓ L)`

Where:
- `ε` (epsilon) controls the perturbation magnitude
- `sign()` extracts only the direction (+1 or -1), not magnitude

### Why it works:
The gradient tells us which direction to push each pixel to **maximize the loss**. By taking the sign, we move each pixel by exactly `ε` in the worst possible direction for the model.

### Implement FGSM attack

The key is to compute gradients with respect to the **input** (not the model parameters).

In [ ]:
def fgsm_attack(model, images, labels, epsilon, device):
    """
    Perform FGSM attack on a batch of images.
    
    Args:
        model: Neural network to attack
        images: Input images (batch)
        labels: True labels
        epsilon: Perturbation magnitude
        device: Device to run on
    
    Returns:
        adversarial_images: Perturbed images
    """
    # Copy images and enable gradient computation
    images = images.clone().detach().to(device)
    images.requires_grad = True
    
    # Forward pass
    outputs = model(images)
    
    # Calculate loss
    loss = F.cross_entropy(outputs, labels)
    
    # Backward pass to get gradients with respect to images
    model.zero_grad()
    loss.backward()
    
    # Get sign of gradients
    gradient_sign = images.grad.sign()
    
    # Create adversarial example
    adversarial_images = images + epsilon * gradient_sign
    
    # Detach from computation graph
    adversarial_images = adversarial_images.detach()
    
    return adversarial_images

### Test FGSM with different epsilon values

Let's see how different perturbation magnitudes affect attack success. Larger epsilon = stronger attack but more visible.

In [ ]:
# Test with different epsilon values
epsilons = [0.0, 0.05, 0.1, 0.2, 0.3]

print("FGSM Attack Results:\n")
print(f"{'Epsilon':<10} {'Clean Acc':<12} {'Attack Acc':<12} {'Success Rate':<12}")
print("-" * 50)

for eps in epsilons:
    correct_clean = 0
    correct_adv = 0
    total = 0
    
    model.eval()
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Get predictions on clean images
        with torch.no_grad():
            outputs_clean = model(images)
            _, pred_clean = outputs_clean.max(1)
            correct_clean += pred_clean.eq(labels).sum().item()
        
        # Generate adversarial examples
        if eps > 0:
            adv_images = fgsm_attack(model, images, labels, eps, device)
        else:
            adv_images = images
        
        # Get predictions on adversarial examples
        with torch.no_grad():
            outputs_adv = model(adv_images)
            _, pred_adv = outputs_adv.max(1)
            correct_adv += pred_adv.eq(labels).sum().item()
        
        total += labels.size(0)
    
    clean_acc = 100. * correct_clean / total
    adv_acc = 100. * correct_adv / total
    success_rate = 100. * (1 - correct_adv / correct_clean)
    
    print(f"{eps:<10.2f} {clean_acc:<12.2f} {adv_acc:<12.2f} {success_rate:<12.2f}")

### Visualize FGSM adversarial examples

Let's see what these adversarial perturbations look like. We'll show:
1. Original images
2. Perturbations (amplified for visibility)
3. Adversarial images

Notice how the perturbations are nearly invisible, yet completely fool the model!

In [ ]:
# Generate adversarial examples for visualization
epsilon = 0.2
test_images, test_labels = next(iter(test_loader))
test_images, test_labels = test_images.to(device), test_labels.to(device)

# Generate adversarial examples
adv_images = fgsm_attack(model, test_images, test_labels, epsilon, device)

# Get predictions
model.eval()
with torch.no_grad():
    outputs_clean = model(test_images)
    _, pred_clean = outputs_clean.max(1)
    
    outputs_adv = model(adv_images)
    _, pred_adv = outputs_adv.max(1)

# Calculate perturbations
perturbations = adv_images - test_images

# Visualize
n_examples = 5
fig, axes = plt.subplots(3, n_examples, figsize=(15, 9))
fig.suptitle(f"FGSM Attack (ε={epsilon})", fontsize=16, fontweight='bold')

for i in range(n_examples):
    # Original image
    img_clean = denormalize(test_images[i]).cpu().squeeze()
    axes[0, i].imshow(img_clean, cmap='gray')
    axes[0, i].set_title(f"Original: {test_labels[i].item()}\nPred: {pred_clean[i].item()}")
    axes[0, i].axis('off')
    
    # Perturbation (amplified by 10x for visibility)
    pert = perturbations[i].cpu().squeeze()
    axes[1, i].imshow(pert * 10, cmap='seismic', vmin=-1, vmax=1)
    axes[1, i].set_title("Perturbation (10×)")
    axes[1, i].axis('off')
    
    # Adversarial image
    img_adv = denormalize(adv_images[i]).cpu().squeeze()
    axes[2, i].imshow(img_adv, cmap='gray')
    color = 'red' if pred_adv[i] != test_labels[i] else 'green'
    axes[2, i].set_title(f"Adversarial\nPred: {pred_adv[i].item()}", color=color)
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

### Key observation:

The perturbations are nearly invisible to the human eye, but they completely change the model's prediction! This is the fundamental vulnerability of neural networks.

The perturbations aren't random noise - they're carefully calculated to push the image across the **decision boundary** of the classifier.

## 6. PGD: Projected Gradient Descent

**PGD** (Projected Gradient Descent) is a stronger, iterative attack. Instead of a single gradient step like FGSM, PGD takes multiple smaller steps.

### The algorithm:
```
x_0 = x  (start with original image)
for t = 1 to num_steps:
    x_t = x_{t-1} + α · sign(∇ₓ L(θ, x_{t-1}, y))
    x_t = clip(x_t, x - ε, x + ε)  # Project back to ε-ball
```

Where:
- `α` (alpha) is the step size (smaller than ε)
- We take multiple steps to find a stronger adversarial example
- The projection ensures we stay within distance `ε` from the original

### Why it's stronger than FGSM:
Multiple steps allow PGD to navigate around the loss surface more effectively, finding better adversarial examples within the same ε-ball constraint.

### Implement PGD attack

We'll add random initialization and multiple gradient steps.

In [ ]:
def pgd_attack(model, images, labels, epsilon, alpha, num_steps, device):
    """
    Perform PGD attack on a batch of images.
    
    Args:
        model: Neural network to attack
        images: Input images (batch)
        labels: True labels
        epsilon: Maximum perturbation (L-infinity norm)
        alpha: Step size for each iteration
        num_steps: Number of PGD iterations
        device: Device to run on
    
    Returns:
        adversarial_images: Perturbed images
    """
    # Clone and move to device
    original_images = images.clone().detach().to(device)
    
    # Start with a random perturbation within epsilon
    adversarial_images = original_images + torch.empty_like(original_images).uniform_(-epsilon, epsilon)
    adversarial_images = torch.clamp(adversarial_images, 0, 1)  # Ensure valid pixel values
    
    # PGD iterations
    for step in range(num_steps):
        adversarial_images.requires_grad = True
        
        # Forward pass
        outputs = model(adversarial_images)
        
        # Calculate loss
        loss = F.cross_entropy(outputs, labels)
        
        # Backward pass
        model.zero_grad()
        loss.backward()
        
        # Get gradient sign
        gradient_sign = adversarial_images.grad.sign()
        
        # Take a step
        adversarial_images = adversarial_images.detach() + alpha * gradient_sign
        
        # Project back to epsilon ball around original image
        perturbation = torch.clamp(adversarial_images - original_images, -epsilon, epsilon)
        adversarial_images = original_images + perturbation
        
        # Ensure valid pixel range
        adversarial_images = torch.clamp(adversarial_images, 0, 1)
    
    return adversarial_images

### Compare PGD vs FGSM attack strength

Let's test PGD with the same epsilon as FGSM to see if multiple iterations produce stronger attacks.

In [ ]:
epsilon = 0.2
alpha = 0.01  # Step size (smaller than epsilon)
num_steps = 40  # Number of iterations

print(f"Attack Comparison (ε={epsilon}):\n")
print(f"{'Attack':<15} {'Accuracy':<12} {'Success Rate':<12}")
print("-" * 40)

# Clean accuracy
correct_clean = 0
total = 0
model.eval()

for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        outputs = model(images)
        _, predicted = outputs.max(1)
        correct_clean += predicted.eq(labels).sum().item()
    total += labels.size(0)

clean_acc = 100. * correct_clean / total
print(f"{'Clean':<15} {clean_acc:<12.2f} {0:<12.2f}")

# FGSM accuracy
correct_fgsm = 0
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    adv_images = fgsm_attack(model, images, labels, epsilon, device)
    with torch.no_grad():
        outputs = model(adv_images)
        _, predicted = outputs.max(1)
        correct_fgsm += predicted.eq(labels).sum().item()

fgsm_acc = 100. * correct_fgsm / total
fgsm_success = 100. * (1 - correct_fgsm / correct_clean)
print(f"{'FGSM':<15} {fgsm_acc:<12.2f} {fgsm_success:<12.2f}")

# PGD accuracy
correct_pgd = 0
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    adv_images = pgd_attack(model, images, labels, epsilon, alpha, num_steps, device)
    with torch.no_grad():
        outputs = model(adv_images)
        _, predicted = outputs.max(1)
        correct_pgd += predicted.eq(labels).sum().item()

pgd_acc = 100. * correct_pgd / total
pgd_success = 100. * (1 - correct_pgd / correct_clean)
print(f"{'PGD':<15} {pgd_acc:<12.2f} {pgd_success:<12.2f}")

print(f"\nPGD is {pgd_success - fgsm_success:.1f}% more effective than FGSM!")

### Visualize PGD adversarial examples

Let's see the difference between FGSM and PGD perturbations.

In [ ]:
# Generate examples
test_images, test_labels = next(iter(test_loader))
test_images, test_labels = test_images.to(device), test_labels.to(device)

fgsm_images = fgsm_attack(model, test_images, test_labels, epsilon, device)
pgd_images = pgd_attack(model, test_images, test_labels, epsilon, alpha, num_steps, device)

# Get predictions
model.eval()
with torch.no_grad():
    _, pred_clean = model(test_images).max(1)
    _, pred_fgsm = model(fgsm_images).max(1)
    _, pred_pgd = model(pgd_images).max(1)

# Visualize
n_examples = 5
fig, axes = plt.subplots(3, n_examples, figsize=(15, 9))
fig.suptitle(f"FGSM vs PGD (ε={epsilon})", fontsize=16, fontweight='bold')

for i in range(n_examples):
    # Original
    img = denormalize(test_images[i]).cpu().squeeze()
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f"Original: {test_labels[i].item()}\nPred: {pred_clean[i].item()}")
    axes[0, i].axis('off')
    
    # FGSM
    img_fgsm = denormalize(fgsm_images[i]).cpu().squeeze()
    axes[1, i].imshow(img_fgsm, cmap='gray')
    color = 'red' if pred_fgsm[i] != test_labels[i] else 'green'
    axes[1, i].set_title(f"FGSM\nPred: {pred_fgsm[i].item()}", color=color)
    axes[1, i].axis('off')
    
    # PGD
    img_pgd = denormalize(pgd_images[i]).cpu().squeeze()
    axes[2, i].imshow(img_pgd, cmap='gray')
    color = 'red' if pred_pgd[i] != test_labels[i] else 'green'
    axes[2, i].set_title(f"PGD\nPred: {pred_pgd[i].item()}", color=color)
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

## 7. Adversarial Training: Learning to Defend

Now that we can generate strong attacks, how do we defend against them?

**Adversarial training** is the most effective defense: train the model on both clean and adversarial examples.

### The algorithm:
```
for each batch (x, y):
    1. Generate adversarial examples: x_adv = PGD(x, y)
    2. Compute loss on adversarial examples: L(θ, x_adv, y)
    3. Update parameters: θ ← θ - η · ∇_θ L
```

### Why it works:
By training on the worst-case examples the model will encounter, we force it to learn **robust features** that are stable under perturbations. The model learns decision boundaries with larger margins.

### Implement adversarial training

We'll modify our training loop to generate adversarial examples on-the-fly during training.

In [ ]:
def adversarial_train_epoch(model, loader, optimizer, criterion, epsilon, alpha, num_steps, device):
    """
    Train for one epoch using adversarial training.
    """
    model.train()
    total_loss = 0
    correct_clean = 0
    correct_adv = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Adv Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # Generate adversarial examples
        model.eval()  # Set to eval mode for attack generation
        adv_images = pgd_attack(model, images, labels, epsilon, alpha, num_steps, device)
        model.train()  # Back to train mode
        
        # Forward pass on adversarial examples
        outputs = model(adv_images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct_adv += predicted.eq(labels).sum().item()
        
        # Also track clean accuracy
        with torch.no_grad():
            outputs_clean = model(images)
            _, pred_clean = outputs_clean.max(1)
            correct_clean += pred_clean.eq(labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    clean_acc = 100. * correct_clean / total
    adv_acc = 100. * correct_adv / total
    
    return avg_loss, clean_acc, adv_acc

### Train a robust model

Let's train a new model from scratch using adversarial training. This will take longer because we generate attacks for every batch.

Note: We'll use a smaller epsilon (0.1) for training stability.

In [ ]:
# Create a new model for adversarial training
robust_model = SimpleCNN().to(device)
robust_optimizer = torch.optim.Adam(robust_model.parameters(), lr=0.001)
robust_criterion = nn.CrossEntropyLoss()

# Adversarial training parameters
train_epsilon = 0.1
train_alpha = 0.01
train_steps = 10  # Fewer steps for faster training

print("Training robust model with adversarial training...\n")
print(f"Attack parameters: ε={train_epsilon}, α={train_alpha}, steps={train_steps}\n")

num_epochs = 3
for epoch in range(num_epochs):
    loss, clean_acc, adv_acc = adversarial_train_epoch(
        robust_model, train_loader, robust_optimizer, robust_criterion,
        train_epsilon, train_alpha, train_steps, device
    )
    
    # Evaluate on test set
    test_clean_acc = evaluate(robust_model, test_loader, device)
    
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {loss:.4f}")
    print(f"  Train: Clean={clean_acc:.2f}%, Adv={adv_acc:.2f}%")
    print(f"  Test:  Clean={test_clean_acc:.2f}%")

### Compare standard vs robust models

Let's test both models against attacks to see how adversarial training improves robustness.

In [ ]:
def evaluate_robustness(model, loader, epsilon, alpha, num_steps, device):
    """Evaluate model on clean and adversarial examples."""
    model.eval()
    correct_clean = 0
    correct_fgsm = 0
    correct_pgd = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # Clean accuracy
        with torch.no_grad():
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct_clean += predicted.eq(labels).sum().item()
        
        # FGSM attack
        fgsm_images = fgsm_attack(model, images, labels, epsilon, device)
        with torch.no_grad():
            outputs = model(fgsm_images)
            _, predicted = outputs.max(1)
            correct_fgsm += predicted.eq(labels).sum().item()
        
        # PGD attack
        pgd_images = pgd_attack(model, images, labels, epsilon, alpha, num_steps, device)
        with torch.no_grad():
            outputs = model(pgd_images)
            _, predicted = outputs.max(1)
            correct_pgd += predicted.eq(labels).sum().item()
        
        total += labels.size(0)
    
    return {
        'clean': 100. * correct_clean / total,
        'fgsm': 100. * correct_fgsm / total,
        'pgd': 100. * correct_pgd / total,
    }

# Evaluate both models
test_epsilon = 0.2
test_alpha = 0.01
test_steps = 40

print(f"Model Comparison (ε={test_epsilon}):\n")

print("Standard Model:")
standard_results = evaluate_robustness(model, test_loader, test_epsilon, test_alpha, test_steps, device)
print(f"  Clean:    {standard_results['clean']:.2f}%")
print(f"  vs FGSM:  {standard_results['fgsm']:.2f}%")
print(f"  vs PGD:   {standard_results['pgd']:.2f}%")

print("\nRobust Model (Adversarially Trained):")
robust_results = evaluate_robustness(robust_model, test_loader, test_epsilon, test_alpha, test_steps, device)
print(f"  Clean:    {robust_results['clean']:.2f}%")
print(f"  vs FGSM:  {robust_results['fgsm']:.2f}%")
print(f"  vs PGD:   {robust_results['pgd']:.2f}%")

print("\nImprovement:")
print(f"  Clean:    {robust_results['clean'] - standard_results['clean']:+.2f}%")
print(f"  vs FGSM:  {robust_results['fgsm'] - standard_results['fgsm']:+.2f}%")
print(f"  vs PGD:   {robust_results['pgd'] - standard_results['pgd']:+.2f}%")

### The accuracy-robustness tradeoff

Notice something important: the robust model has **lower clean accuracy** but **much higher adversarial accuracy**.

This is called the **accuracy-robustness tradeoff**: making a model robust often sacrifices some performance on clean data. The model learns more conservative decision boundaries that are farther from the data points.

This is a fundamental limitation - you can't have perfect accuracy AND perfect robustness simultaneously.

## 8. Certified Defenses: Provable Robustness

All the defenses we've seen so far are **empirical** - they work against known attacks but don't provide guarantees against future attacks.

**Certified defenses** aim to provide **provable robustness**: mathematical guarantees that no attack within a certain threat model can succeed.

### Main approaches:

1. **Randomized Smoothing**
   - Add Gaussian noise to inputs during inference
   - Provably robust within a radius proportional to noise level
   - Works by making the model's decision boundary "fuzzy"
   - Trade-off: larger certified radius = lower clean accuracy

2. **Interval Bound Propagation (IBP)**
   - During training, compute bounds on all intermediate activations
   - Guarantee predictions within these bounds
   - Very conservative but provides hard guarantees

3. **Abstract Interpretation**
   - Use formal verification techniques from program analysis
   - Verify properties of neural networks mathematically
   - Computationally expensive but provides strong guarantees

### The fundamental limitation:
There's a **certified accuracy vs certified radius** tradeoff. The larger the perturbation you want to certify against, the lower your accuracy will be.

Current state-of-the-art certified defenses can only handle small perturbations (ε ≈ 0.1-0.5 for MNIST/CIFAR-10), and they significantly reduce clean accuracy.

### Visualize the concept of certified robustness

Let's create a simple 2D visualization to understand certified vs empirical defenses.

In [ ]:
# Create a simple 2D visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Generate synthetic 2D data
np.random.seed(42)
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)

# Standard model: tight decision boundary
Z1 = np.sign(X + 0.5 * Y - 0.2 + 0.3 * np.sin(3 * X) + 0.2 * np.cos(3 * Y))
axes[0].contourf(X, Y, Z1, levels=[-1, 0, 1], colors=['lightblue', 'lightcoral'], alpha=0.6)
axes[0].contour(X, Y, Z1, levels=[0], colors='black', linewidths=2)
axes[0].scatter([1], [1], c='red', s=200, marker='*', edgecolors='black', linewidth=2, zorder=5, label='Data point')
circle1 = plt.Circle((1, 1), 0.3, color='red', fill=False, linewidth=2, linestyle='--', label='ε-ball')
axes[0].add_patch(circle1)
axes[0].set_title('Standard Model\n(Vulnerable)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)

# Adversarially trained model: wider margin
Z2 = np.sign(X + 0.5 * Y - 0.2)
axes[1].contourf(X, Y, Z2, levels=[-1, 0, 1], colors=['lightblue', 'lightcoral'], alpha=0.6)
axes[1].contour(X, Y, Z2, levels=[0], colors='black', linewidths=2)
axes[1].scatter([1], [1], c='red', s=200, marker='*', edgecolors='black', linewidth=2, zorder=5)
circle2 = plt.Circle((1, 1), 0.3, color='red', fill=False, linewidth=2, linestyle='--')
axes[1].add_patch(circle2)
axes[1].set_title('Adversarially Trained\n(Empirically Robust)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)

# Certified defense: guaranteed safe region
Z3 = np.sign(X + 0.5 * Y)
axes[2].contourf(X, Y, Z3, levels=[-1, 0, 1], colors=['lightblue', 'lightcoral'], alpha=0.6)
axes[2].contour(X, Y, Z3, levels=[0], colors='black', linewidths=2)
axes[2].scatter([1], [1], c='red', s=200, marker='*', edgecolors='black', linewidth=2, zorder=5)
circle3 = plt.Circle((1, 1), 0.3, color='green', fill=False, linewidth=3, label='Certified safe')
axes[2].add_patch(circle3)
axes[2].set_title('Certified Defense\n(Provably Robust)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(-3, 3)
axes[2].set_ylim(-3, 3)

plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Standard: Decision boundary is close to data, can cross the ε-ball")
print("- Adversarially Trained: Smoother boundary, empirically stays outside ε-ball")
print("- Certified: Guaranteed that entire ε-ball has same prediction (green circle)")

## 9. Real-World Deployment Considerations

Adversarial robustness isn't just an academic problem - it has serious implications for deploying ML systems in the real world.

### Critical Applications:

1. **Autonomous Vehicles**
   - Stop sign misclassification from stickers (Eykholt et al., 2018)
   - Traffic light spoofing
   - Lane detection attacks
   - **Impact**: Life-threatening accidents

2. **Face Recognition / Biometric Security**
   - Adversarial glasses that fool face recognition (Sharif et al., 2016)
   - Makeup patterns that evade detection
   - Fingerprint spoofing
   - **Impact**: Security breaches, unauthorized access

3. **Malware Detection**
   - Adding benign code to evade detection
   - Modifying bytes to change ML predictions
   - **Impact**: Malware goes undetected

4. **Medical Diagnosis**
   - Adversarial examples in CT/MRI scans (Finlayson et al., 2019)
   - Misdiagnosis of tumors, diseases
   - **Impact**: Wrong treatments, patient harm

5. **Spam/Fraud Detection**
   - Crafted emails/messages that evade filters
   - Credit card fraud disguised as legitimate
   - **Impact**: Financial losses

### Practical Defense Strategies

When deploying ML systems, consider these defense-in-depth approaches:

1. **Adversarial Training**
   - Train on worst-case examples
   - Best empirical defense available
   - Cost: Increased training time, lower clean accuracy

2. **Input Preprocessing**
   - JPEG compression (removes small perturbations)
   - Bit depth reduction
   - Random resizing/padding
   - Warning: Can be circumvented by adaptive attacks

3. **Ensemble Defenses**
   - Use multiple models with different architectures
   - Attacks must fool all models simultaneously (harder)
   - Cost: Computational overhead

4. **Detection Methods**
   - Monitor for unusual input patterns
   - Use auxiliary detectors
   - Statistical tests on activations
   - Warning: Can have high false positive rates

5. **Certified Defenses** (when guarantees needed)
   - For high-stakes applications
   - Accept accuracy loss for guaranteed safety
   - Currently only practical for small perturbations

6. **Defense in Depth**
   - Don't rely on ML model alone
   - Add redundant verification systems
   - Human oversight for critical decisions
   - Anomaly detection and monitoring

### The Arms Race

Adversarial ML is an **arms race** between attackers and defenders:

1. **Defenders** propose a defense
2. **Attackers** find adaptive attacks that break it
3. **Defenders** propose improved defenses
4. Repeat...

Many published defenses have been broken by stronger attacks. The only consistently effective defenses are:
- **Adversarial training** (best empirical defense)
- **Certified defenses** (provable but limited)

### Key Takeaways for Practitioners:

1. **Assume adversaries exist** in security-critical applications
2. **Test against strong attacks** (PGD, not just FGSM)
3. **Use adversarial training** if robustness is important
4. **Monitor deployed systems** for unusual inputs
5. **Don't rely on obscurity** - assume attackers know your model
6. **Consider certified defenses** when you need guarantees
7. **Defense in depth** - ML should be one layer in a larger security system

## 10. Key Takeaways

### What We Learned:

1. **Adversarial Vulnerability is Fundamental**
   - Neural networks are surprisingly fragile to imperceptible perturbations
   - Even state-of-the-art models are vulnerable
   - This isn't a bug - it's a fundamental property of high-dimensional linear models

2. **Attack Methods**
   - **FGSM**: Single gradient step, fast but weaker
   - **PGD**: Multiple steps, slower but much stronger
   - PGD is the "gold standard" for testing robustness

3. **Adversarial Training Works**
   - Training on adversarial examples significantly improves robustness
   - Trade-off: Lower clean accuracy for higher adversarial accuracy
   - Currently the most effective empirical defense

4. **Certified Defenses Provide Guarantees**
   - Mathematical proof of robustness within a radius
   - More conservative than adversarial training
   - Limited to small perturbations currently

5. **Real-World Implications**
   - Critical for security-sensitive applications
   - Autonomous vehicles, face recognition, malware detection, medical diagnosis
   - Requires defense-in-depth approach

### The Big Picture:

Adversarial examples reveal that **what neural networks learn is not what we think they learn**. They don't necessarily extract the same features humans use. They can rely on imperceptible patterns that are fragile under small perturbations.

This is a **fundamental challenge** for AI safety and reliability. As we deploy ML systems in critical applications, understanding and defending against adversarial examples becomes essential.

### Further Reading:

- Goodfellow et al. (2015): "Explaining and Harnessing Adversarial Examples"
- Madry et al. (2018): "Towards Deep Learning Models Resistant to Adversarial Attacks"
- Cohen et al. (2019): "Certified Adversarial Robustness via Randomized Smoothing"
- Carlini & Wagner (2017): "Towards Evaluating the Robustness of Neural Networks"
- Ilyas et al. (2019): "Adversarial Examples Are Not Bugs, They Are Features"